In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
import numpy as np
import random
import os
from sklearn.metrics import classification_report

# 1. SETUP & CONFIG
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

n_qubits = 10
num_classes = 26
batch_size = 16
num_epochs = 50

# Robust Training Hyperparameters
EPSILON_Q = 0.1       # Feature space perturbation scale (will be re-calc'd)
EPS_FGSM = 0.03
EPS_PGD = 0.1
ALPHA_PGD = 0.01
ITERS_PGD = 7

# 2. DATA LOADING (MaleVis settings)
TRAIN_PATH = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/train'
VAL_PATH   = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/val'

# Transforms from Base Model
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH, transform=eval_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    print(f"** Dataset Loaded: {len(train_dataset)} Train, {len(val_dataset)} Val **")
except Exception as e:
    print(f"Error loading datasets: {e}")
    # Fallback for code verification
    print("Creating dummy loaders...")
    os.makedirs("dummy_data/train/class1", exist_ok=True)
    os.makedirs("dummy_data/val/class1", exist_ok=True)
    train_dataset = ImageFolder("dummy_data/train", transform=train_transform)
    val_dataset   = ImageFolder("dummy_data/val", transform=eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 3. QUANTUM LAYER
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    
    # Basic Entangler Layers
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# 4. MODEL ARCHITECTURE
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),    
            nn.BatchNorm2d(8), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),   
            nn.BatchNorm2d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 224, 3, stride=2, padding=1), 
            nn.BatchNorm2d(224), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                
        )
        self.fc_expand = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)

class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)
        q_out = self.q_layer(x) 
        return self.classifier(q_out)

# 5. HELPER FUNCTIONS: CENTROIDS, EPSILON, PERTURBATION

def compute_quantum_epsilon(model, n_cnots=30, depth=6, alpha=1.0, beta=1.0):
    """
    Compute epsilon_q based on quantum circuit complexity.
    """
    epsilon_q = 1.0 / (1 + alpha * n_cnots + beta * depth)
    return epsilon_q

def compute_class_centroids(model, loader, device, num_classes):
    """
    Compute class centroids in the FEATURE SPACE (before quantum layer).
    """
    model.eval()
    sums = torch.zeros(num_classes, n_qubits, device=device)
    counts = torch.zeros(num_classes, device=device)
    
    print("Computing class centroids...")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            features = model.feature_extractor(x)
            features = torch.tanh(features)
            
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sums[c] += features[mask].sum(0)
                    counts[c] += mask.sum()
    
    counts[counts == 0] = 1
    centroids = sums / counts.unsqueeze(1)
    return centroids

def qni_ccp_feature_perturbation_fixed(model, x, y, centroids, epsilon_q=0.1, target_class=None):
    """
    QNI-CCP Feature Perturbation Logic
    """
    model.eval()
    
    # Get original features
    with torch.no_grad():
        original_features = model.feature_extractor(x)
        original_features = torch.tanh(original_features)
    
    # Create copy for gradient calculation
    perturbed_features = original_features.clone().detach().requires_grad_(True)
    
    # Forward through Quantum + Classifier
    try:
        q_out = model.q_layer(perturbed_features)
    except:
        q_out = torch.stack([model.q_layer(f) for f in perturbed_features])
        
    logits = model.classifier(q_out)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    
    # Get Sensitivity (S)
    if perturbed_features.grad is None:
        S = torch.zeros_like(perturbed_features)
    else:
        S = perturbed_features.grad.data
    
    # Select Target Class (Centroid)
    if target_class is None:
        target_classes = []
        for i in range(y.size(0)):
            available_classes = [c for c in range(centroids.size(0)) if c != y[i].item()]
            if available_classes:
                target_classes.append(random.choice(available_classes))
            else:
                target_classes.append((y[i].item() + 1) % centroids.size(0))
        target_class = torch.tensor(target_classes, device=y.device)
    else:
        target_class = torch.full_like(y, target_class)
    
    mu_c_prime = centroids[target_class]
    
    # Compute Perturbation: epsilon * S * (mu' - feature)
    perturbation_direction = mu_c_prime - original_features
    weighted_perturbation = S * perturbation_direction 
    
    perturbed_features_final = original_features + epsilon_q * weighted_perturbation
    return perturbed_features_final.detach()

# 6. ADVERSARIAL ATTACKS (FGSM & PGD)

def fgsm_attack(model, images, labels, eps_fgsm=0.03):
    model.eval()
    images_adv = images.clone().detach().to(device).requires_grad_(True)
    labels = labels.to(device)

    logits = model(images_adv)
    loss = F.cross_entropy(logits, labels)
    model.zero_grad()
    loss.backward()

    images_adv = images_adv + eps_fgsm * images_adv.grad.sign()
    images_adv = torch.clamp(images_adv, -1.0, 1.0)
    return images_adv.detach()

def pgd_attack(model, images, labels, pgd_eps=0.1, pgd_alpha=0.01, pgd_iters=7):
    model.eval()
    images_orig = images.clone().detach().to(device)
    labels = labels.to(device)

    images_adv = images_orig + torch.empty_like(images_orig).uniform_(-pgd_eps, pgd_eps)
    images_adv = torch.clamp(images_adv, -1.0, 1.0).detach()

    for _ in range(pgd_iters):
        images_adv.requires_grad_(True)
        logits = model(images_adv)
        loss = F.cross_entropy(logits, labels)
        model.zero_grad()
        loss.backward()

        perturb = pgd_alpha * images_adv.grad.sign()
        images_adv = images_adv + perturb
        
        delta = torch.clamp(images_adv - images_orig, min=-pgd_eps, max=pgd_eps)
        images_adv = torch.clamp(images_orig + delta, -1.0, 1.0).detach()

    return images_adv

# 7. EVALUATION HELPER

def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0

# 8. ROBUST TRAINING LOOP

def train_with_feature_perturbation_and_adv(model,
                                            train_loader,
                                            val_loader,
                                            centroids,
                                            epsilon_q,
                                            epsilon_fgsm=0.03,
                                            eps_pgd=0.1,
                                            alpha_pgd=0.01,
                                            iters_pgd=7,
                                            num_epochs=50):
    
    opt   = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=5)
    best_val_acc = 0.0

    print(f"Starting Robust Training for {num_epochs} epochs...")

    for epoch in range(1, num_epochs + 1):
        # Update centroids every 5 epochs
        if epoch % 5 == 0:
            centroids = compute_class_centroids(model, train_loader, device, num_classes)

        model.train()
        running_loss = 0.0
        running_corr = 0
        running_total = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)

        for xb, yb in loop:
            xb, yb = xb.to(device), yb.to(device)

            # --- 1) Clean Loss ---
            logits_clean = model(xb)
            loss_clean   = F.cross_entropy(logits_clean, yb)

            # --- 2) QNI-CCP Loss (Feature Perturbation) ---
            perturbed_features = qni_ccp_feature_perturbation_fixed(
                model, xb, yb, centroids, epsilon_q=epsilon_q
            )
            # Pass perturbed features through Quantum + Classifier
            try:
                q_out = model.q_layer(perturbed_features)
            except:
                q_out = torch.stack([model.q_layer(f) for f in perturbed_features])
            
            logits_qni = model.classifier(q_out)
            loss_qni   = F.cross_entropy(logits_qni, yb)

            # --- 3) FGSM Loss ---
            xb_fgsm     = fgsm_attack(model, xb, yb, eps_fgsm=epsilon_fgsm)
            logits_fgsm = model(xb_fgsm)
            loss_fgsm   = F.cross_entropy(logits_fgsm, yb)

            # --- 4) PGD Loss ---
            xb_pgd      = pgd_attack(model, xb, yb, pgd_eps=eps_pgd, pgd_alpha=alpha_pgd, pgd_iters=iters_pgd)
            logits_pgd  = model(xb_pgd)
            loss_pgd    = F.cross_entropy(logits_pgd, yb)

            # --- 5) Centroid Regularization ---
            # Pull clean features closer to their class centroid
            current_features = torch.tanh(model.feature_extractor(xb))
            batch_centroids  = centroids[yb]
            centroid_reg     = F.mse_loss(current_features, batch_centroids)

            # --- Combined Loss ---
            # Weighted sum of all components
            loss = (
                0.5  * loss_clean +
                0.15 * loss_qni   +
                0.1  * loss_fgsm  +
                0.15 * loss_pgd   +
                0.1  * centroid_reg
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            running_loss += loss.item() * xb.size(0)
            running_corr += (logits_clean.argmax(1) == yb).sum().item()
            running_total += xb.size(0)
            
            loop.set_postfix(loss=loss.item())

        # Epoch Metrics
        train_loss = running_loss / running_total
        train_acc  = running_corr / running_total
        
        sched.step(train_loss)

        # Validation
        val_acc = evaluate(model, val_loader)
        print(f"Epoch {epoch:2d} | Train L: {train_loss:.4f} | Train A: {train_acc:.4f} | Val A: {val_acc:.4f}")

        # Save Best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "adv2.pth")
            print("  💾 Saved best model.")

    return best_val_acc

# 9. MAIN BLOCK
if __name__ == "__main__":
    # Initialize Model
    model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)
    
    # Pre-compute Epsilon
    # n_cnots approx 6 layers * (6 qubits - 1) = 30 CNOTs
    eps_q = compute_quantum_epsilon(model, n_cnots=30, depth=6)
    print(f"Computed Quantum Epsilon: {eps_q:.4f}")
    
    # Pre-compute Centroids
    centroids = compute_class_centroids(model, train_loader, device, num_classes)
    
    # Start Training
    best_acc = train_with_feature_perturbation_and_adv(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        centroids=centroids,
        epsilon_q=eps_q,
        epsilon_fgsm=EPS_FGSM,
        eps_pgd=EPS_PGD,
        alpha_pgd=ALPHA_PGD,
        iters_pgd=ITERS_PGD,
        num_epochs=num_epochs
    )
    
    print(f"Training Complete. Best Validation Accuracy: {best_acc:.4f}")

Using device: cuda
** Dataset Loaded: 9947 Train, 2149 Val **
Computed Quantum Epsilon: 0.0270
Computing class centroids...
Starting Robust Training for 50 epochs...


Epoch  1 | Train L: 2.8925 | Train A: 0.0548 | Val A: 0.1401
  💾 Saved best model.


Epoch  2 | Train L: 2.4725 | Train A: 0.1403 | Val A: 0.3378
  💾 Saved best model.


Epoch  3 | Train L: 2.1974 | Train A: 0.2280 | Val A: 0.2852


Epoch  4 | Train L: 1.9699 | Train A: 0.3098 | Val A: 0.3965
  💾 Saved best model.
Computing class centroids...


Epoch  5 | Train L: 1.7963 | Train A: 0.3737 | Val A: 0.3951


Epoch 6:   0%|          | 2/622 [00:03<18:31,  1.79s/it, loss=1.42]

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
import numpy as np
import random
import os
from sklearn.metrics import classification_report

# 1. SETUP & CONFIG
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

n_qubits = 10
num_classes = 26
batch_size = 16
num_epochs = 50 # Set to 50 as requested

# Robust Training Hyperparameters
EPSILON_Q = 0.1       
EPS_FGSM = 0.03
EPS_PGD = 0.1
ALPHA_PGD = 0.01
ITERS_PGD = 7

# 2. DATA LOADING
# Update these paths if necessary
TRAIN_PATH = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/train'
VAL_PATH   = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/val'

train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH, transform=eval_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    print(f"** Dataset Loaded: {len(train_dataset)} Train, {len(val_dataset)} Val **")
except Exception as e:
    print(f"Error loading datasets: {e}")
    # Dummy fallback for compilation check
    os.makedirs("dummy_data/train/class1", exist_ok=True)
    os.makedirs("dummy_data/val/class1", exist_ok=True)
    train_dataset = ImageFolder("dummy_data/train", transform=train_transform)
    val_dataset   = ImageFolder("dummy_data/val", transform=eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 3. QUANTUM LAYER
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    
    # Basic Entangler Layers
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# 4. MODEL ARCHITECTURE (Must match saved model exactly)
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),    
            nn.BatchNorm2d(8), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),   
            nn.BatchNorm2d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 224, 3, stride=2, padding=1), 
            nn.BatchNorm2d(224), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                 
        )
        self.fc_expand = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)

class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)
        q_out = self.q_layer(x) 
        return self.classifier(q_out)

# 5. HELPER FUNCTIONS
def compute_quantum_epsilon(model, n_cnots=30, depth=6, alpha=1.0, beta=1.0):
    epsilon_q = 1.0 / (1 + alpha * n_cnots + beta * depth)
    return epsilon_q

def compute_class_centroids(model, loader, device, num_classes):
    model.eval()
    sums = torch.zeros(num_classes, n_qubits, device=device)
    counts = torch.zeros(num_classes, device=device)
    
    print("Computing class centroids...")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            features = model.feature_extractor(x)
            features = torch.tanh(features)
            
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sums[c] += features[mask].sum(0)
                    counts[c] += mask.sum()
    
    counts[counts == 0] = 1
    centroids = sums / counts.unsqueeze(1)
    return centroids

def qni_ccp_feature_perturbation_fixed(model, x, y, centroids, epsilon_q=0.1, target_class=None):
    model.eval()
    with torch.no_grad():
        original_features = model.feature_extractor(x)
        original_features = torch.tanh(original_features)
    
    perturbed_features = original_features.clone().detach().requires_grad_(True)
    
    try:
        q_out = model.q_layer(perturbed_features)
    except:
        q_out = torch.stack([model.q_layer(f) for f in perturbed_features])
        
    logits = model.classifier(q_out)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    
    if perturbed_features.grad is None:
        S = torch.zeros_like(perturbed_features)
    else:
        S = perturbed_features.grad.data
    
    if target_class is None:
        target_classes = []
        for i in range(y.size(0)):
            available_classes = [c for c in range(centroids.size(0)) if c != y[i].item()]
            if available_classes:
                target_classes.append(random.choice(available_classes))
            else:
                target_classes.append((y[i].item() + 1) % centroids.size(0))
        target_class = torch.tensor(target_classes, device=y.device)
    else:
        target_class = torch.full_like(y, target_class)
    
    mu_c_prime = centroids[target_class]
    perturbation_direction = mu_c_prime - original_features
    weighted_perturbation = S * perturbation_direction 
    
    perturbed_features_final = original_features + epsilon_q * weighted_perturbation
    return perturbed_features_final.detach()

def fgsm_attack(model, images, labels, eps_fgsm=0.03):
    model.eval()
    images_adv = images.clone().detach().to(device).requires_grad_(True)
    labels = labels.to(device)

    logits = model(images_adv)
    loss = F.cross_entropy(logits, labels)
    model.zero_grad()
    loss.backward()

    images_adv = images_adv + eps_fgsm * images_adv.grad.sign()
    images_adv = torch.clamp(images_adv, -1.0, 1.0)
    return images_adv.detach()

def pgd_attack(model, images, labels, pgd_eps=0.1, pgd_alpha=0.01, pgd_iters=7):
    model.eval()
    images_orig = images.clone().detach().to(device)
    labels = labels.to(device)

    images_adv = images_orig + torch.empty_like(images_orig).uniform_(-pgd_eps, pgd_eps)
    images_adv = torch.clamp(images_adv, -1.0, 1.0).detach()

    for _ in range(pgd_iters):
        images_adv.requires_grad_(True)
        logits = model(images_adv)
        loss = F.cross_entropy(logits, labels)
        model.zero_grad()
        loss.backward()

        perturb = pgd_alpha * images_adv.grad.sign()
        images_adv = images_adv + perturb
        
        delta = torch.clamp(images_adv - images_orig, min=-pgd_eps, max=pgd_eps)
        images_adv = torch.clamp(images_orig + delta, -1.0, 1.0).detach()

    return images_adv

def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0

# 8. RESUME TRAINING LOOP
def train_with_feature_perturbation_and_adv(model,
                                            train_loader,
                                            val_loader,
                                            centroids,
                                            epsilon_q,
                                            epsilon_fgsm=0.03,
                                            eps_pgd=0.1,
                                            alpha_pgd=0.01,
                                            iters_pgd=7,
                                            num_epochs=50,
                                            save_path="adv2_continued.pth"): # Modified to save to new file
    
    # Re-initialize optimizer (standard practice when resuming without saving optimizer state)
    opt   = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=5)
    
    # Calculate initial accuracy to ensure we don't save a worse model initially
    print("Evaluating initial performance...")
    best_val_acc = evaluate(model, val_loader)
    print(f"Initial Validation Accuracy (from loaded model): {best_val_acc:.4f}")

    print(f"Starting Robust Training (Resume) for {num_epochs} epochs...")

    for epoch in range(1, num_epochs + 1):
        if epoch % 5 == 0:
            centroids = compute_class_centroids(model, train_loader, device, num_classes)

        model.train()
        running_loss = 0.0
        running_corr = 0
        running_total = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)

        for xb, yb in loop:
            xb, yb = xb.to(device), yb.to(device)

            # 1) Clean Loss
            logits_clean = model(xb)
            loss_clean   = F.cross_entropy(logits_clean, yb)

            # 2) QNI-CCP
            perturbed_features = qni_ccp_feature_perturbation_fixed(
                model, xb, yb, centroids, epsilon_q=epsilon_q
            )
            try:
                q_out = model.q_layer(perturbed_features)
            except:
                q_out = torch.stack([model.q_layer(f) for f in perturbed_features])
            
            logits_qni = model.classifier(q_out)
            loss_qni   = F.cross_entropy(logits_qni, yb)

            # 3) FGSM
            xb_fgsm     = fgsm_attack(model, xb, yb, eps_fgsm=epsilon_fgsm)
            logits_fgsm = model(xb_fgsm)
            loss_fgsm   = F.cross_entropy(logits_fgsm, yb)

            # 4) PGD
            xb_pgd      = pgd_attack(model, xb, yb, pgd_eps=eps_pgd, pgd_alpha=alpha_pgd, pgd_iters=iters_pgd)
            logits_pgd  = model(xb_pgd)
            loss_pgd    = F.cross_entropy(logits_pgd, yb)

            # 5) Centroid Regularization
            current_features = torch.tanh(model.feature_extractor(xb))
            batch_centroids  = centroids[yb]
            centroid_reg     = F.mse_loss(current_features, batch_centroids)

            loss = (
                0.5  * loss_clean +
                0.15 * loss_qni   +
                0.1  * loss_fgsm  +
                0.15 * loss_pgd   +
                0.1  * centroid_reg
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            running_loss += loss.item() * xb.size(0)
            running_corr += (logits_clean.argmax(1) == yb).sum().item()
            running_total += xb.size(0)
            
            loop.set_postfix(loss=loss.item())

        train_loss = running_loss / running_total
        train_acc  = running_corr / running_total
        
        sched.step(train_loss)

        val_acc = evaluate(model, val_loader)
        print(f"Epoch {epoch:2d} | Train L: {train_loss:.4f} | Train A: {train_acc:.4f} | Val A: {val_acc:.4f}")

        # Save Best (Compare against previous best)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  💾 Saved best model to {save_path}")

    return best_val_acc

# 9. MAIN BLOCK (RESUME LOGIC)
if __name__ == "__main__":
    # 1. Initialize Model Structure
    print("Initializing model architecture...")
    model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)
    
    # 2. LOAD PREVIOUS WEIGHTS
    load_path = "adv2.pth"
    if os.path.exists(load_path):
        print(f"🔄 Loading weights from {load_path} to resume training...")
        model.load_state_dict(torch.load(load_path, map_location=device))
    else:
        print(f"⚠️  WARNING: {load_path} not found! Starting training from scratch.")

    # 3. Pre-compute Parameters (Needed for the training loop)
    eps_q = compute_quantum_epsilon(model, n_cnots=30, depth=6)
    print(f"Computed Quantum Epsilon: {eps_q:.4f}")
    
    # Recalculate centroids based on the LOADED model
    centroids = compute_class_centroids(model, train_loader, device, num_classes)
    
    # 4. Resume Training
    print("🚀 Resuming training...")
    best_acc = train_with_feature_perturbation_and_adv(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        centroids=centroids,
        epsilon_q=eps_q,
        epsilon_fgsm=EPS_FGSM,
        eps_pgd=EPS_PGD,
        alpha_pgd=ALPHA_PGD,
        iters_pgd=ITERS_PGD,
        num_epochs=num_epochs,  # This will run for 50 MORE epochs
        save_path="adv2_continued.pth" # Save to a new file to stay safe
    )
    
    print(f"Continued Training Complete. Best Validation Accuracy: {best_acc:.4f}")

Using device: cuda
** Dataset Loaded: 9947 Train, 2149 Val **
Initializing model architecture...
🔄 Loading weights from adv2.pth to resume training...
Computed Quantum Epsilon: 0.0270
Computing class centroids...
🚀 Resuming training...
Evaluating initial performance...
Initial Validation Accuracy (from loaded model): 0.8153
Starting Robust Training (Resume) for 50 epochs...


Epoch  1 | Train L: 0.4281 | Train A: 0.9157 | Val A: 0.8032


Epoch  2 | Train L: 0.4163 | Train A: 0.9195 | Val A: 0.8078


Epoch  3 | Train L: 0.4079 | Train A: 0.9208 | Val A: 0.8064


Epoch  4 | Train L: 0.4146 | Train A: 0.9202 | Val A: 0.8167
  💾 Saved best model to adv2_continued.pth
Computing class centroids...


RuntimeError: DataLoader worker (pid(s) 2847302, 2847303, 2847304, 2847305) exited unexpectedly